In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
from scipy.stats import chi2
import warnings

warnings.filterwarnings("ignore")


###**CÁC BƯỚC XỬ LÍ FILE DU LIỆU GỐC :**

BƯỚC 1: ĐỌC DỮ LIỆU GỐC

BƯỚC 2: LỌC 4 NHÓM ORIGIN

BƯỚC 3: KIỂM TRA & XỬ LÝ MISSING

BƯỚC 4: PHÁT HIỆN OUTLIER BẰNG MAHALANOBIS

BƯỚC 5: KIỂM TRA DỮ LIỆU SAU LÀM SẠCH


## 1. ĐỌC DỮ LIỆU GỐC

In [2]:
df = pd.read_csv(Path("..") / "data" / "raw" / "arabica_data_cleaned.csv")
print(f"Kích thước dữ liệu ban đầu : {df.shape}")
print(f"Số dòng                     : {len(df)}")
print(f"Số cột                      : {len(df.columns)}")

Kích thước dữ liệu ban đầu : (1311, 44)
Số dòng                     : 1311
Số cột                      : 44


## 2.CHỌN 7 BIẾN DÙNG CHO MANOVA


In [3]:
selected_columns = [
    "Aroma",
    "Flavor",
    "Aftertaste",
    "Acidity",
    "Body",
    "Moisture",
    "Country.of.Origin"
]

# Kiểm tra cột tồn tại
missing_cols = [c for c in selected_columns if c not in df.columns]
if missing_cols:
    raise ValueError(f"Các cột sau không tồn tại trong dữ liệu: {missing_cols}")

df_manova = df[selected_columns].copy()
print("\nCác biến được chọn:")
print(df_manova.columns.tolist())


Các biến được chọn:
['Aroma', 'Flavor', 'Aftertaste', 'Acidity', 'Body', 'Moisture', 'Country.of.Origin']


##3.ĐỔI TÊN BIẾN

In [4]:
df_manova.rename(
    columns={
        "Aroma": "Fragrance",
        "Country.of.Origin": "Origin"
    },
    inplace=True
)

print("\nTên biến sau khi đổi:")
print(df_manova.columns.tolist())



Tên biến sau khi đổi:
['Fragrance', 'Flavor', 'Aftertaste', 'Acidity', 'Body', 'Moisture', 'Origin']


4. CHỈ GIỮ 4 NHÓM ORIGIN

In [5]:
selected_origins = ["Mexico", "Colombia", "Guatemala", "Brazil"]

df_manova = df_manova[df_manova["Origin"].isin(selected_origins)].copy()
df_manova.reset_index(drop=True, inplace=True)
print("\nSố lượng mẫu theo nhóm:")
print(df_manova["Origin"].value_counts().to_string())
print(f"\nTổng số mẫu sau khi lọc: {len(df_manova)}")


Số lượng mẫu theo nhóm:
Origin
Mexico       236
Colombia     183
Guatemala    181
Brazil       132

Tổng số mẫu sau khi lọc: 732


##5. CHUYỂN 6 BIẾN LIÊN TỤC SANG NUMERIC

In [6]:
dependent_vars = [
    "Fragrance", "Flavor", "Aftertaste",
    "Acidity", "Body", "Moisture"
]

for col in dependent_vars:
    df_manova[col] = pd.to_numeric(df_manova[col], errors="coerce")


## 6. KIỂM TRA & XỬ LÝ MISSING

In [7]:
print("\nMissing trước khi xử lý:")
print(df_manova.isnull().sum().to_string())

before_missing = len(df_manova)

# Loại bỏ dòng có missing ở biến phụ thuộc hoặc Origin
df_manova = df_manova.dropna(subset=dependent_vars + ["Origin"]).copy()
df_manova.reset_index(drop=True, inplace=True)

after_missing = len(df_manova)

print(f"\nSố dòng trước xử lý missing : {before_missing}")
print(f"Số dòng bị loại do missing  : {before_missing - after_missing}")
print(f"Số dòng sau xử lý missing   : {after_missing}")

print("\nMissing sau xử lý:")
print(df_manova.isnull().sum().to_string())

print("\nSố lượng mẫu từng Origin sau xử lý missing:")
print(df_manova["Origin"].value_counts().to_string())



Missing trước khi xử lý:
Fragrance     0
Flavor        0
Aftertaste    0
Acidity       0
Body          0
Moisture      0
Origin        0

Số dòng trước xử lý missing : 732
Số dòng bị loại do missing  : 0
Số dòng sau xử lý missing   : 732

Missing sau xử lý:
Fragrance     0
Flavor        0
Aftertaste    0
Acidity       0
Body          0
Moisture      0
Origin        0

Số lượng mẫu từng Origin sau xử lý missing:
Origin
Mexico       236
Colombia     183
Guatemala    181
Brazil       132


##7. PHÁT HIỆN OUTLIER BẰNG MAHALANOBIS (vector hóa)

In [8]:
X = df_manova[dependent_vars].to_numpy(dtype=float)

# Vector trung bình
mean_vector = np.mean(X, axis=0)

# Ma trận hiệp phương sai
cov_matrix = np.cov(X, rowvar=False)

# Kiểm tra khả nghịch
if np.linalg.matrix_rank(cov_matrix) < cov_matrix.shape[0]:
    raise ValueError(
        "Ma trận hiệp phương sai không khả nghịch. "
        "Không thể tính khoảng cách Mahalanobis."
    )

# Nghịch đảo
cov_inv = np.linalg.inv(cov_matrix)

# Tính Mahalanobis D² (vector hóa - nhanh hơn)
diff = X - mean_vector
mahalanobis_d2 = np.einsum('ij,jk,ik->i', diff, cov_inv, diff)
# Cách tương đương dễ hiểu hơn:
# mahalanobis_d2 = np.sum(diff @ cov_inv * diff, axis=1)

df_manova["Mahalanobis_D2"] = mahalanobis_d2

# ============================================================
# 9. XÁC ĐỊNH NGƯỠNG OUTLIER
# ============================================================
p = len(dependent_vars)          # số biến
alpha = 0.001                    # mức ý nghĩa rất nghiêm ngặt
threshold = chi2.ppf(1 - alpha, df=p)

print(f"\nSố biến dùng tính Mahalanobis : {p}")
print(f"Alpha                         : {alpha}")
print(f"Ngưỡng Chi-square (χ²)        : {threshold:.4f}")



Số biến dùng tính Mahalanobis : 6
Alpha                         : 0.001
Ngưỡng Chi-square (χ²)        : 22.4577


##8. XÁC ĐỊNH NGƯỠNG OUTLIER

In [9]:
p = len(dependent_vars)          # số biến
alpha = 0.001                    # mức ý nghĩa rất nghiêm ngặt
threshold = chi2.ppf(1 - alpha, df=p)

print(f"\nSố biến dùng tính Mahalanobis : {p}")
print(f"Alpha                         : {alpha}")
print(f"Ngưỡng Chi-square (χ²)        : {threshold:.4f}")


Số biến dùng tính Mahalanobis : 6
Alpha                         : 0.001
Ngưỡng Chi-square (χ²)        : 22.4577


##9. XÁC ĐỊNH & LOẠI OUTLIER

In [10]:
outlier_mask = df_manova["Mahalanobis_D2"] > threshold
n_outliers = outlier_mask.sum()

print(f"\nSố outlier phát hiện: {n_outliers}")

if n_outliers > 0:
    print("\nMột số outlier tiêu biểu (top 5 theo D²):")
    print(
        df_manova.loc[outlier_mask]
        .nlargest(5, "Mahalanobis_D2")
        [["Origin"] + dependent_vars + ["Mahalanobis_D2"]]
        .round(3)
        .to_string()
    )

before_outlier = len(df_manova)
df_clean = df_manova[~outlier_mask].copy()
df_clean.reset_index(drop=True, inplace=True)
after_outlier = len(df_clean)

print(f"\nSố dòng trước khi loại outlier : {before_outlier}")
print(f"Số outlier bị loại             : {before_outlier - after_outlier}")
print(f"Số dòng sau khi loại outlier   : {after_outlier}")



Số outlier phát hiện: 12

Một số outlier tiêu biểu (top 5 theo D²):
        Origin  Fragrance  Flavor  Aftertaste  Acidity  Body  Moisture  Mahalanobis_D2
366   Colombia       5.08    7.75        7.83     7.75  7.83      0.11         171.082
493   Colombia       7.75    7.75        7.75     7.67  5.25      0.00         145.255
558   Colombia       7.67    7.50        7.42     5.25  7.67      0.00         130.562
683  Guatemala       7.17    6.08        6.17     7.25  7.33      0.11          36.390
700  Guatemala       6.42    6.50        6.50     7.33  6.33      0.12          30.148

Số dòng trước khi loại outlier : 732
Số outlier bị loại             : 12
Số dòng sau khi loại outlier   : 720


##10. XÓA CỘT MAHALANOBIS & SẮP XẾP CỘT

In [11]:
df_clean = df_clean.drop(columns=["Mahalanobis_D2"])

df_clean = df_clean[
    ["Fragrance", "Flavor", "Aftertaste",
     "Acidity", "Body", "Moisture", "Origin"]
]

##11. KIỂM TRA DỮ LIỆU CUỐI

In [12]:
print(f"\nKích thước cuối: {df_clean.shape}")
print("\nMissing:")
print(df_clean.isnull().sum().to_string())

print("\nSố lượng từng nhóm Origin:")
print(df_clean["Origin"].value_counts().to_string())

print("\nThống kê mô tả 6 biến liên tục:")
print(df_clean[dependent_vars].describe().round(3).to_string())

print("\n5 dòng đầu:")
print(df_clean.head().to_string())


Kích thước cuối: (720, 7)

Missing:
Fragrance     0
Flavor        0
Aftertaste    0
Acidity       0
Body          0
Moisture      0
Origin        0

Số lượng từng nhóm Origin:
Origin
Mexico       236
Colombia     180
Guatemala    175
Brazil       129

Thống kê mô tả 6 biến liên tục:
       Fragrance   Flavor  Aftertaste  Acidity     Body  Moisture
count    720.000  720.000     720.000  720.000  720.000   720.000
mean       7.555    7.502       7.358    7.521    7.494     0.093
std        0.275    0.300       0.317    0.266    0.253     0.046
min        6.500    6.330       6.170    6.750    6.670     0.000
25%        7.420    7.330       7.170    7.330    7.330     0.100
50%        7.580    7.580       7.420    7.500    7.500     0.110
75%        7.750    7.670       7.580    7.670    7.670     0.120
max        8.580    8.500       8.420    8.500    8.330     0.220

5 dòng đầu:
   Fragrance  Flavor  Aftertaste  Acidity  Body  Moisture     Origin
0       8.42    8.50        8.42     8.

## 12. XUẤT FILE EXCEL

In [13]:
output_path = Path("..") / "data" / "processed" / "clean_data.xlsx"
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_excel(output_path, index=False, sheet_name="data_clean")
print("HOÀN THÀNH")
print(f"File dữ liệu sạch đã được tạo tại:\n{output_path}")
print(f"\nTổng số mẫu cuối cùng: {len(df_clean)}")

HOÀN THÀNH
File dữ liệu sạch đã được tạo tại:
..\data\processed\clean_data.xlsx

Tổng số mẫu cuối cùng: 720
